# 📱 liveclip on your phone (free)

Clip a **live** YouTube or Twitch stream the moment its chat explodes. It runs on Google's free computers, so you only need a phone and a Google account. No laptop, no install.

**How to use it**
1. Paste the live stream link in the box below (YouTube app: **Share → Copy link**).
2. Tap the **▶ play button** next to the box. If Colab says *"This notebook was not authored by Google"*, tap **Run anyway**.
3. Keep this tab open. Every time the chat spikes, a vertical 9:16 clip with captions shows up below with a **Download** button. A status line every minute shows the chat speed, so you know it's working.

Tap the **■ stop button** any time; clips that are still being made get finished first.

> **Only clip streamers who allow clipping** (or who run an official clipping program). Clips without permission get copyright strikes and no payout.

Made by [@aaditya.knows](https://instagram.com/aaditya.knows) · code: [github.com/Aadi1659/liveclip](https://github.com/Aadi1659/liveclip)

In [ ]:
#@title ▶ Paste the live link, then tap play
stream_url = ""  #@param {type:"string"}
starting_time = "00:00:00"  #@param {type:"string"}
minutes = 60  #@param {type:"slider", min:5, max:1440, step:5}
max_time = "24h"  #@param ["use minutes slider", "30m", "1h", "2h", "4h", "8h", "12h", "24h"]
max_clips = 5  #@param {type:"slider", min:1, max:20, step:1}
clip_length = "30 sec"  #@param ["20 sec", "30 sec", "45 sec", "60 sec"]
how_many_clips = "normal"  #@param ["more clips", "normal", "fewer clips"]
captions = "hinglish"  #@param ["hinglish", "english", "off"]
save_to_google_drive = False  #@param {type:"boolean"}

import os, re, signal, subprocess, sys, base64, shutil
from IPython.display import display, HTML

url = stream_url.strip()
if not re.search(r"(youtube\.com|youtu\.be|twitch\.tv)/", url):
    raise SystemExit("Paste a YouTube or Twitch LIVE link in the stream_url box first.")

# 1) setup (first run only, ~1 minute)
if not os.path.exists("/content/liveclip/liveclip.py"):
    print("Setting up (first run only, about a minute)…")
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/Aadi1659/liveclip.git", "/content/liveclip"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlink>=7.0", "faster-whisper>=1.0", "pillow>=10.0"], check=True)
    print("Setup done.")
else:
    subprocess.run(["git", "-C", "/content/liveclip", "pull", "-q"])  # pick up fixes on a reused runtime

drive_dir = None
if save_to_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = "/content/drive/MyDrive/liveclip"; os.makedirs(drive_dir, exist_ok=True)

secs = int(clip_length.split()[0])
spike = {"more clips": "1.4", "normal": "1.7", "fewer clips": "2.2"}[how_many_clips]
out_dir = "/content/clips"
cmd = [sys.executable, "-u", "/content/liveclip/liveclip.py", url, "--out", out_dir,
       "--pre", str(secs - 6), "--post", "6", "--spike", spike,
       "--duration", str(minutes * 60), "--max-time", max_time,
       "--start-time", starting_time, "--max-clips", str(max_clips),
       "--captions", captions, "--model", "auto", "--device", "auto"]


def show(path):
    """Show the clip in the notebook with a player you can download from."""
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    name = os.path.basename(path)
    display(HTML(f'<p style="font:600 15px sans-serif;margin:14px 0 6px">🎬 {name}</p>'
                 f'<video controls playsinline style="width:100%;max-width:360px;border-radius:12px" '
                 f'src="data:video/mp4;base64,{b64}"></video>'
                 f'<br><a download="{name}" href="data:video/mp4;base64,{b64}" '
                 f'style="display:inline-block;margin-top:6px;padding:10px 16px;border-radius:10px;'
                 f'background:#e8684a;color:#fff;font:600 15px sans-serif;text-decoration:none">⬇ Download {name}</a>'))
    if drive_dir:
        shutil.copy(path, drive_dir); print(f"   saved to Google Drive → My Drive/liveclip/{name}")


print("Starting. Clips appear below as the chat spikes. Tap ■ to stop.\n")
env = dict(os.environ, HF_HUB_VERBOSITY="error", HF_HUB_DISABLE_PROGRESS_BARS="1", PYTHONWARNINGS="ignore")
# own session: Colab's stop button only reaches this cell, which then asks liveclip to finish its open clips
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, start_new_session=True)
stopping = False
while True:
    try:
        line = p.stdout.readline()
        if not line:
            if p.poll() is not None: break
            continue
        line = line.rstrip()
        if "unauthenticated requests" in line or "[ctranslate2]" in line: continue
        print(line)
        m = re.search(r"saved (\S+\.mp4)$", line)
        if m and os.path.exists(m.group(1)): show(m.group(1))
    except KeyboardInterrupt:
        if not stopping:
            stopping = True; print("\nStopping… finishing the clips that are being made.")
            p.send_signal(signal.SIGINT)
        else:
            p.kill(); break
print("\nDone. If you lost this page, run the next cell to see every clip again.")

In [ ]:
#@title 🎞 Show all my clips again
import glob, os, base64
from IPython.display import display, HTML
clips = sorted(glob.glob("/content/clips/*.mp4"))
if not clips: print("No clips yet. Run the cell above first.")
for path in clips:
    b64 = base64.b64encode(open(path, "rb").read()).decode(); name = os.path.basename(path)
    display(HTML(f'<p style="font:600 15px sans-serif;margin:14px 0 6px">🎬 {name}</p>'
                 f'<video controls playsinline style="width:100%;max-width:360px;border-radius:12px" src="data:video/mp4;base64,{b64}"></video>'
                 f'<br><a download="{name}" href="data:video/mp4;base64,{b64}" style="display:inline-block;margin-top:6px;'
                 f'padding:10px 16px;border-radius:10px;background:#e8684a;color:#fff;font:600 15px sans-serif;text-decoration:none">⬇ Download {name}</a>'))

### Tips
- **No clips after 10 minutes?** Pick *more clips*. Big streams already have fast chat, so it rarely doubles. **Too many?** Pick *fewer clips*.
- **Want it faster?** Menu → **Runtime → Change runtime type → T4 GPU** before you tap play. Captions then take a couple of seconds instead of ~30.
- **Phone screen locked or tab closed?** The clips are still on the Colab computer for a while. Come back and run *Show all my clips again*, or tick **save_to_google_drive** next time so every clip also lands in your Drive.
- A chat spike means *something* happened, not that the clip is good. Watch each one before you post it. Captions can have mistakes; fix them in Edits or CapCut.
- Free Colab sessions stop after a few hours or when idle. That's Google's limit, not this tool's.